## Prompt Optimization with MLflow

### Installing Utilities and Libraries

In [1]:
%pip install langchain-anthropic==1.5.4 langchain==1.3.14 langgraph==1.2.10 mlflow==3.15.2 dspy==3.3.1

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
anthropic_api_key = os.getenv("CLAUDE_API_KEY")
anthropic_model_name = os.getenv("CLAUDE_MODEL_NAME")

os.environ["ANTHROPIC_API_KEY"] = anthropic_api_key

### Instantiating the ChatAnthropic Class

In [3]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key,
)

### Enable MLflow Tracing

In [4]:
import mlflow

# Calling autolog for LangChain will enable trace logging.
mlflow.langchain.autolog()

# Optional: Set a tracking URI and an experiment
mlflow.set_experiment("prompt-optimization")
mlflow.set_tracking_uri("http://localhost:5001")

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Register the Initial Prompt

In [5]:
PROMPT_NAME = "ticket-classification-prompt-optimization"

prompt = mlflow.genai.register_prompt(
    name=PROMPT_NAME,

    template="""
Classify the following IT support ticket.

Ticket:
{{ticket}}
""",

    commit_message="v1: Basic ticket classification prompt"
)


# set a production alias
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="development",
    version=1
)

print(f"Prompt: {prompt.name}")
print(f"Version: {prompt.version}")

2026/09/27 15:42:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: ticket-classification-prompt-optimization, version 1


Prompt: ticket-classification-prompt-optimization
Version: 1


### Define the Prediction Function

In [6]:
from langchain_core.messages import HumanMessage

def predict_fn(query):

        prompt = mlflow.genai.load_prompt(
            name_or_uri=(
                f"prompts:/{PROMPT_NAME}@development"
            )
        )

        formatted_prompt = prompt.format(
            ticket=query
        )

        response = model.invoke(
                [HumanMessage(content=formatted_prompt)]
        )

        return response.content

### Test your Prediction Function

In [7]:
ticket_query = "My second monitor occasionally flickers, but I can continue working normally"

output = predict_fn(query = ticket_query)


print(output)

# IT Support Ticket Classification

**Category:** Hardware Issue

**Subcategory:** Display/Monitor

**Priority:** Low

**Impact:** Minor (non-blocking)

**Urgency:** Low

**Status:** Open - Pending Investigation

---

### Details:

| Field | Value |
|---|---|
| **Issue Type** | Intermittent Hardware Malfunction |
| **Affected Component** | Secondary Monitor |
| **User Impact** | Minimal - user can continue working |
| **Business Impact** | Low - no work stoppage reported |
| **Recommended Action** | Check cable connections, test with different port/cable, update graphics drivers |
| **SLA Target** | Standard (3-5 business days) |

### Suggested Troubleshooting Steps:
1. Check and reseat video cable (HDMI/DisplayPort/VGA) connections
2. Try a different cable or port on the same monitor
3. Test the monitor on another machine (or test another monitor on this machine) to isolate hardware vs. driver issue
4. Update/reinstall graphics drivers
5. Check monitor refresh rate settings
6. Inspect

### Optimize against Data

In [8]:
# Training data with inputs, expected outputs, and expectations
dataset = [
    {
        "inputs": {
            "query": (
                "I changed my password this morning and "
                "now I cannot log into my corporate account."
            )
        },
        "outputs": {
            "response": "Category: Authentication\nPriority: High"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Authentication'",
                "Priority must be 'High'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "The office Wi-Fi is unavailable for everyone "
                "on the third floor and nobody can access "
                "internal applications."
            )
        },
        "outputs": {
            "response": "Category: Network\nPriority: Critical"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Network'",
                "Priority must be 'Critical'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "Microsoft Excel crashes whenever I try to "
                "open one particular spreadsheet. Other "
                "spreadsheets work normally."
            )
        },
        "outputs": {
            "response": "Category: Software\nPriority: Medium"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Software'",
                "Priority must be 'Medium'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "I received an email asking me to enter my "
                "company password on an unfamiliar website."
            )
        },
        "outputs": {
            "response": "Category: Security\nPriority: High"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Security'",
                "Priority must be 'High'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "My second monitor occasionally flickers, "
                "but I can continue working normally."
            )
        },
        "outputs": {
            "response": "Category: Hardware\nPriority: Low"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Hardware'",
                "Priority must be 'Low'"
            ]
        }
    }
]

Trace(trace_id=tr-247efbfce9a46d0a786f43c99b0f9dc7)

In [9]:
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Correctness

# Optimize the prompt
result = mlflow.genai.optimize_prompts(
    predict_fn = predict_fn,
    train_data = dataset,
    prompt_uris = [prompt.uri],
    optimizer = GepaPromptOptimizer(reflection_model = "anthropic:/claude-opus-4-5"),
    scorers = [Correctness(model="anthropic:/claude-opus-4-5")]
)

# Use the Optimized Prompt
optimized_prompt = result.optimized_prompts[0]
print(f"Optimized Template: {optimized_prompt.template}")

2026/09/27 15:44:47 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/09/27 15:44:48 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
/usr/local/lib/python3.10/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


Iteration 0: Base program full valset score: 0.0 over 5 / 5 examples
Iteration 1: Selected program 0 score: 0.0
Iteration 1: Proposed new text for ticket-classification-prompt-optimization: Classify the following IT support ticket by providing exactly two fields:

1. **Category** - Must be exactly one of these values:
   - "Hardware" - Physical equipment issues (monitors, keyboards, printers, computers, etc.)
   - "Software" - Application issues (crashes, errors, installation problems, etc.)
   - "Network" - Connectivity issues (Wi-Fi, internet, VPN, internal network access, etc.)

2. **Priority** - Must be exactly one of these values:
   - "Low" - Minor issues where the user can continue working normally
   - "Medium" - Issues that block specific functionality but don't stop all work
   - "Critical" - Issues affecting multiple users or preventing all work

Output format:
Category: [category]
Priority: [priority]

Do not include sub-categories, troubleshooting steps, or additional deta

Iteration 3: Selected program 2 score: 1.0
Iteration 3: All subsample scores perfect for parent 2. Skipping.
Iteration 3: Reflective mutation did not propose a new candidate
Iteration 4: Selected program 2 score: 1.0
Iteration 4: All subsample scores perfect for parent 2. Skipping.
Iteration 4: Reflective mutation did not propose a new candidate
Iteration 5: Selected program 2 score: 1.0
Iteration 5: All subsample scores perfect for parent 2. Skipping.
Iteration 5: Reflective mutation did not propose a new candidate
Iteration 6: Selected program 2 score: 1.0
Iteration 6: All subsample scores perfect for parent 2. Skipping.
Iteration 6: Reflective mutation did not propose a new candidate
Iteration 7: Selected program 2 score: 1.0
Iteration 7: All subsample scores perfect for parent 2. Skipping.
Iteration 7: Reflective mutation did not propose a new candidate
Iteration 8: Selected program 2 score: 1.0
Iteration 8: All subsample scores perfect for parent 2. Skipping.
Iteration 8: Reflecti

2026/09/27 15:48:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: ticket-classification-prompt-optimization, version 2


Iteration 27: Selected program 2 score: 1.0
Iteration 27: All subsample scores perfect for parent 2. Skipping.
Iteration 27: Reflective mutation did not propose a new candidate
🏃 View run able-deer-552 at: http://localhost:5001/#/experiments/5/runs/5692614417704e4e8b97257a9c0c5758
🧪 View experiment at: http://localhost:5001/#/experiments/5
Optimized Template: Classify the following IT support ticket by providing exactly two fields:

1. **Category** - Must be exactly one of these values:
   - "Hardware" - Physical equipment issues (monitors, keyboards, printers, computers, docking stations, etc.)
   - "Software" - Application issues (crashes, errors, installation problems, bugs, performance issues, etc.)
   - "Network" - Connectivity issues (Wi-Fi, internet, VPN, internal network access, slow connections, etc.)
   - "Authentication" - Login and access issues (password problems, account lockouts, unable to log in, credential issues, SSO problems)
   - "Security" - Security threats and co

### Use the Optimized Prompt

In [11]:
VERSION_NUMBER = "LATEST_PROMPT_VERSION_NUMBER"

def predict_fn(query):

        prompt = mlflow.genai.load_prompt(
            name_or_uri=(
                f"prompts:/{PROMPT_NAME}/{VERSION_NUMBER}"
            )
        )

        formatted_prompt = prompt.format(
            ticket=query
        )

        response = model.invoke(
                [HumanMessage(content=formatted_prompt)]
        )

        return response.content

In [12]:
ticket_query = "My second monitor occasionally flickers, but I can continue working normally"

output = predict_fn(query = ticket_query)


print(output)

ValueError: invalid literal for int() with base 10: 'LATEST_PROMPT_VERSION_NUMBER'